# Why SVD? LDA on High-Dimensional Face Data (N < D)

**Objective:** Demonstrate why the SVD solver is essential for Linear Discriminant Analysis
when the number of samples is smaller than the number of features.

**Key insight:** The within-class scatter matrix $S_W$ is at most rank $N - C$ (where $N$ = samples,
$C$ = classes). When $N < D$ (features), $S_W$ is singular and cannot be inverted directly.
The eigen and LSQR solvers require a full-rank covariance matrix (or shrinkage to regularize it),
while the SVD solver sidesteps this by working in the sample space.

**Dataset:** Olivetti Faces — 400 grayscale images (64x64 = 4,096 features) of 40 subjects.
With only 10 images per subject, $N \ll D$, making the covariance matrix heavily rank-deficient.

In [ ]:
from __future__ import annotations

import warnings
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

from sklearn.datasets import fetch_olivetti_faces
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

## 1. Load and inspect the Olivetti Faces dataset

Each face is a 64x64 grayscale image flattened to 4,096 features.
With 400 total samples and 40 classes, the within-class scatter matrix
has rank at most $400 - 40 = 360$, far below the 4,096 features.

In [ ]:
faces = fetch_olivetti_faces()
X = faces.data.astype(np.float64)
y = faces.target
images = faces.images

n_samples, n_features = X.shape
n_classes = len(np.unique(y))
max_cov_rank = n_samples - n_classes

print(f"Samples (N):        {n_samples}")
print(f"Features (D):       {n_features}  ({int(np.sqrt(n_features))}x{int(np.sqrt(n_features))} pixels)")
print(f"Classes (C):        {n_classes}  (subjects)")
print(f"Samples per class:  {n_samples // n_classes}")
print(f"Max rank of S_W:    {max_cov_rank}  (N - C)")
print(f"Rank deficit:       {n_features - max_cov_rank}  (D - rank)")
print(f"\nN < D:  {n_samples} < {n_features}  =>  Covariance matrix is SINGULAR")

In [ ]:
# Show sample faces (one per subject)
fig, axes = plt.subplots(4, 10, figsize=(14, 5.6))
for i, ax in enumerate(axes.flat):
    idx = i * (n_samples // n_classes)  # first image of each subject
    ax.imshow(images[idx], cmap='gray')
    ax.set_title(f"#{i}", fontsize=8)
    ax.axis('off')
fig.suptitle('Olivetti Faces: one sample per subject (40 subjects)', y=1.01)
plt.tight_layout()
plt.show()

## 2. The rank-deficiency problem

Let's verify that the pooled within-class covariance matrix is indeed singular.
We compute it explicitly and examine its eigenvalue spectrum.

In [ ]:
# Compute pooled within-class covariance
class_means = np.zeros((n_classes, n_features))
for c in range(n_classes):
    class_means[c] = X[y == c].mean(axis=0)

Xc = np.empty_like(X)
for c in range(n_classes):
    Xc[y == c] = X[y == c] - class_means[c]

S_W = (Xc.T @ Xc) / n_samples
eigvals = np.linalg.eigvalsh(S_W)
eigvals_sorted = eigvals[::-1]  # descending

n_nonzero = np.sum(eigvals_sorted > 1e-10)
print(f"Covariance matrix shape: {S_W.shape}")
print(f"Non-zero eigenvalues:    {n_nonzero} / {n_features}")
print(f"Zero eigenvalues:        {n_features - n_nonzero}")
print(f"Expected rank:           {max_cov_rank} (N - C)")
print(f"\nCondition number of non-zero part: {eigvals_sorted[0] / eigvals_sorted[n_nonzero - 1]:.1f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Full spectrum
ax1.semilogy(np.arange(1, n_features + 1), np.maximum(eigvals_sorted, 1e-20), lw=1.2)
ax1.axvline(max_cov_rank, color='red', ls='--', lw=1.5, label=f'rank = N-C = {max_cov_rank}')
ax1.set_xlabel('Eigenvalue index')
ax1.set_ylabel('Eigenvalue (log scale)')
ax1.set_title('Eigenvalue spectrum of $S_W$')
ax1.legend()
ax1.grid(alpha=0.3)

# Zoomed around the rank boundary
window = 40
lo, hi = max(0, max_cov_rank - window), min(n_features, max_cov_rank + window)
ax2.semilogy(
    np.arange(lo + 1, hi + 1),
    np.maximum(eigvals_sorted[lo:hi], 1e-20),
    'o-', ms=3, lw=1.2,
)
ax2.axvline(max_cov_rank, color='red', ls='--', lw=1.5, label=f'rank = {max_cov_rank}')
ax2.set_xlabel('Eigenvalue index')
ax2.set_ylabel('Eigenvalue (log scale)')
ax2.set_title('Zoom: eigenvalue cliff at rank boundary')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"The eigenvalues drop to machine zero after index {max_cov_rank}.")
print(f"Inverting this matrix is impossible without regularization.")

## 3. Solver comparison: SVD vs eigen vs LSQR

We now compare the three LDA solvers on this rank-deficient problem.

- **SVD**: Works in the sample space via SVD of the centered data matrix. Never forms or inverts the covariance matrix. Handles singularity naturally.
- **Eigen**: Solves the generalized eigenvalue problem $S_B w = \lambda S_W w$. Requires inverting $S_W$, which fails when singular (needs shrinkage).
- **LSQR**: Solves via least-squares on the covariance. Also requires a non-singular $S_W$ (needs shrinkage).

In [ ]:
SEED = 42
TEST_SIZE = 0.25

splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y))
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Train samples per class: ~{X_train.shape[0] // n_classes}")
print(f"N_train ({X_train.shape[0]}) << D ({n_features}): covariance is singular")

In [ ]:
results = []

configs = [
    ('svd',  None, 'SVD (no shrinkage needed)'),
    ('eigen', None, 'Eigen (no shrinkage)'),
    ('eigen', 'auto', 'Eigen (shrinkage=auto)'),
    ('eigen', 0.1, 'Eigen (shrinkage=0.1)'),
    ('lsqr', None, 'LSQR (no shrinkage)'),
    ('lsqr', 'auto', 'LSQR (shrinkage=auto)'),
    ('lsqr', 0.1, 'LSQR (shrinkage=0.1)'),
]

for solver, shrinkage, label in configs:
    clf = LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage)
    error = None
    acc = np.nan
    t = np.nan
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            t0 = perf_counter()
            clf.fit(X_train, y_train)
            t = perf_counter() - t0
            acc = clf.score(X_test, y_test)
    except Exception as e:
        error = type(e).__name__ + ': ' + str(e)[:80]

    results.append({
        'label': label,
        'solver': solver,
        'shrinkage': shrinkage,
        'accuracy': acc,
        'time_s': t,
        'error': error,
    })

# Print results table
print(f"{'Config':<30} {'Accuracy':>10} {'Time (s)':>10} {'Error'}")
print('-' * 85)
for r in results:
    acc_str = f"{r['accuracy']:.4f}" if np.isfinite(r['accuracy']) else 'FAILED'
    t_str = f"{r['time_s']:.4f}" if np.isfinite(r['time_s']) else '-'
    err_str = r['error'] if r['error'] else ''
    print(f"{r['label']:<30} {acc_str:>10} {t_str:>10} {err_str}")

In [ ]:
# Bar chart of accuracies
valid = [r for r in results if np.isfinite(r['accuracy'])]
failed = [r for r in results if not np.isfinite(r['accuracy'])]

fig, ax = plt.subplots(figsize=(10, 5))
labels = [r['label'] for r in valid]
accs = [r['accuracy'] for r in valid]
colors = ['#2196F3' if r['solver'] == 'svd' else '#FF9800' if r['solver'] == 'eigen' else '#4CAF50'
          for r in valid]

bars = ax.barh(labels, accs, color=colors, edgecolor='white', height=0.6)
ax.set_xlim(0, 1.05)
ax.set_xlabel('Test accuracy')
ax.set_title('LDA solver comparison on Olivetti Faces (N=300, D=4096)')

for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{acc:.3f}', va='center', fontsize=10)

if failed:
    ax.text(0.5, -0.12,
            'Failed configs: ' + ', '.join(r['label'] for r in failed),
            transform=ax.transAxes, fontsize=9, style='italic', color='red',
            ha='center')

ax.grid(alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

## 4. Batch vs streaming parity (SVD solver)

Since only the SVD solver works without shrinkage on this data, we verify that
`partial_fit` (streaming) achieves the same accuracy as `fit` (batch).
This is the core use case: incremental face recognition where new face images
arrive over time and the model updates without re-training from scratch.

In [ ]:
# With 40 classes, very small chunk sizes risk not seeing all classes early.
# We use chunk sizes >= 50 so all classes appear within the first chunk or two.
STREAM_CHUNK_SIZES = [50, 75, 100, 150, 300]

# Batch baseline
clf_batch = LinearDiscriminantAnalysis(solver='svd')
t0 = perf_counter()
clf_batch.fit(X_train, y_train)
batch_time = perf_counter() - t0
batch_acc = clf_batch.score(X_test, y_test)
batch_preds = clf_batch.predict(X_test)
batch_proba = clf_batch.predict_proba(X_test)

print(f"Batch SVD: accuracy={batch_acc:.4f}, time={batch_time:.4f}s")
print()

# Streaming at various chunk sizes
classes = np.unique(y_train)
stream_results = []

for chunk_size in STREAM_CHUNK_SIZES:
    clf_stream = LinearDiscriminantAnalysis(solver='svd')
    t0 = perf_counter()
    for start in range(0, len(X_train), chunk_size):
        end = min(start + chunk_size, len(X_train))
        clf_stream.partial_fit(
            X_train[start:end],
            y_train[start:end],
            classes=classes if start == 0 else None,
        )
    stream_time = perf_counter() - t0
    stream_acc = clf_stream.score(X_test, y_test)
    stream_preds = clf_stream.predict(X_test)
    stream_proba = clf_stream.predict_proba(X_test)

    pred_agreement = np.mean(stream_preds == batch_preds)
    proba_max_diff = np.max(np.abs(stream_proba - batch_proba))

    stream_results.append({
        'chunk_size': chunk_size,
        'accuracy': stream_acc,
        'time_s': stream_time,
        'pred_agreement': pred_agreement,
        'proba_max_diff': proba_max_diff,
    })

print(f"{'Chunk':>6} {'Accuracy':>10} {'Agreement':>11} {'Max P diff':>12} {'Time (s)':>10}")
print('-' * 55)
print(f"{'batch':>6} {batch_acc:>10.4f} {'(ref)':>11} {'(ref)':>12} {batch_time:>10.4f}")
for r in stream_results:
    print(f"{r['chunk_size']:>6} {r['accuracy']:>10.4f} {r['pred_agreement']:>10.4f}% {r['proba_max_diff']:>12.2e} {r['time_s']:>10.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

chunks = [r['chunk_size'] for r in stream_results]

# Accuracy
ax = axes[0]
ax.plot(chunks, [r['accuracy'] for r in stream_results], 'o-', lw=2, label='streaming')
ax.axhline(batch_acc, color='gray', ls='--', lw=1.5, label='batch')
ax.set_xlabel('Chunk size')
ax.set_ylabel('Test accuracy')
ax.set_title('Accuracy: batch vs streaming')
ax.set_xscale('log')
ax.legend()
ax.grid(alpha=0.3)

# Prediction agreement
ax = axes[1]
ax.plot(chunks, [r['pred_agreement'] for r in stream_results], 's-', lw=2, color='tab:green')
ax.axhline(1.0, color='gray', ls='--', lw=1.5)
ax.set_xlabel('Chunk size')
ax.set_ylabel('Prediction agreement with batch')
ax.set_title('Prediction parity')
ax.set_xscale('log')
ax.set_ylim(0.95, 1.005)
ax.grid(alpha=0.3)

# Training time
ax = axes[2]
ax.plot(chunks, [r['time_s'] for r in stream_results], 'D-', lw=2, color='tab:orange', label='streaming')
ax.axhline(batch_time, color='gray', ls='--', lw=1.5, label='batch')
ax.set_xlabel('Chunk size')
ax.set_ylabel('Training time (s)')
ax.set_title('Training time')
ax.set_xscale('log')
ax.legend()
ax.grid(alpha=0.3)

fig.suptitle('SVD partial_fit parity on Olivetti Faces (N < D)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Visualizing LDA projections

LDA finds at most $C - 1 = 39$ discriminant directions. We project the test faces
onto the first two discriminant components and compare batch vs streaming.

In [ ]:
# Use one streaming model (chunk=75) for comparison
clf_stream_viz = LinearDiscriminantAnalysis(solver='svd')
for start in range(0, len(X_train), 75):
    end = min(start + 75, len(X_train))
    clf_stream_viz.partial_fit(
        X_train[start:end], y_train[start:end],
        classes=classes if start == 0 else None,
    )

X_proj_batch = clf_batch.transform(X_test)
X_proj_stream = clf_stream_viz.transform(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pick a subset of classes for readability
show_classes = np.arange(0, 10)
cmap = plt.cm.tab10

for ax, X_proj, title in [
    (ax1, X_proj_batch, 'Batch fit'),
    (ax2, X_proj_stream, 'Streaming partial_fit (chunk=75)'),
]:
    for ci, c in enumerate(show_classes):
        mask = y_test == c
        if mask.sum() > 0:
            ax.scatter(
                X_proj[mask, 0], X_proj[mask, 1],
                c=[cmap(ci)], s=40, alpha=0.7, label=f'Subject {c}',
                edgecolors='white', linewidths=0.5,
            )
    ax.set_xlabel('LD1')
    ax.set_ylabel('LD2')
    ax.set_title(title)
    ax.grid(alpha=0.2)

ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, ncol=1)
fig.suptitle('LDA projection of Olivetti Faces (first 10 subjects shown)', y=1.01)
plt.tight_layout()
plt.show()

## 6. Streaming with shrinkage-based solvers for comparison

The eigen and LSQR solvers can handle this data *if* shrinkage is applied.
Let's compare all three solvers in streaming mode with appropriate regularization,
highlighting that SVD achieves strong results without any hyperparameter tuning.

In [ ]:
# With 40 classes, we need a chunk large enough that all classes appear
# within the first few chunks. chunk_size=75 ensures this.
CHUNK_SIZE = 75

solver_configs = [
    ('svd', {}, 'SVD'),
    ('lsqr', {'shrinkage': 0.01}, 'LSQR (0.01)'),
    ('lsqr', {'shrinkage': 0.1}, 'LSQR (0.1)'),
    ('eigen', {'shrinkage': 0.01}, 'Eigen (0.01)'),
    ('eigen', {'shrinkage': 0.1}, 'Eigen (0.1)'),
]

comparison = []
for solver, params, label in solver_configs:
    # Batch
    clf_b = LinearDiscriminantAnalysis(solver=solver, **params)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        clf_b.fit(X_train, y_train)
    acc_b = clf_b.score(X_test, y_test)

    # Streaming
    clf_s = LinearDiscriminantAnalysis(solver=solver, **params)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for start in range(0, len(X_train), CHUNK_SIZE):
            end = min(start + CHUNK_SIZE, len(X_train))
            clf_s.partial_fit(
                X_train[start:end], y_train[start:end],
                classes=classes if start == 0 else None,
            )
    acc_s = clf_s.score(X_test, y_test)

    preds_b = clf_b.predict(X_test)
    preds_s = clf_s.predict(X_test)
    agreement = np.mean(preds_b == preds_s)

    comparison.append({
        'label': label,
        'batch_acc': acc_b,
        'stream_acc': acc_s,
        'agreement': agreement,
    })

print(f"{'Solver':<18} {'Batch Acc':>10} {'Stream Acc':>12} {'Agreement':>11}")
print('-' * 55)
for r in comparison:
    print(f"{r['label']:<18} {r['batch_acc']:>10.4f} {r['stream_acc']:>12.4f} {r['agreement']:>10.4f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(comparison))
width = 0.35
labels = [r['label'] for r in comparison]

batch_accs = [r['batch_acc'] for r in comparison]
stream_accs = [r['stream_acc'] for r in comparison]

bars1 = ax.bar(x - width / 2, batch_accs, width, label='Batch', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x + width / 2, stream_accs, width, label='Streaming', color='#FF9800', alpha=0.8)

ax.set_ylabel('Test accuracy')
ax.set_title(f'Batch vs Streaming accuracy (chunk_size={CHUNK_SIZE})')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.legend()
ax.set_ylim(0.7, 1.0)
ax.grid(alpha=0.2, axis='y')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Misclassified faces

Let's look at which faces the SVD batch model gets wrong — a sanity check that
errors are on genuinely ambiguous images.

In [ ]:
preds = clf_batch.predict(X_test)
wrong = np.where(preds != y_test)[0]

n_show = min(len(wrong), 12)
if n_show > 0:
    fig, axes = plt.subplots(2, min(n_show, 6), figsize=(14, 5))
    axes = axes.flat if n_show > 1 else [axes]
    for i, ax in enumerate(axes):
        if i < n_show:
            idx = wrong[i]
            ax.imshow(X_test[idx].reshape(64, 64), cmap='gray')
            ax.set_title(f'True={y_test[idx]}, Pred={preds[idx]}', fontsize=9)
        ax.axis('off')
    fig.suptitle(f'Misclassified faces ({len(wrong)} of {len(y_test)} test samples)', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Perfect classification on the test set!')

print(f"\nTest accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Misclassified: {len(wrong)} / {len(y_test)}")

## Summary

1. **Rank deficiency is inherent** to high-dimensional data: with 400 samples and 4,096 features, the within-class covariance matrix has rank at most 360 and 3,736 zero eigenvalues.

2. **SVD solver handles this naturally** by decomposing the centered data matrix directly, never forming or inverting the $D \times D$ covariance. No shrinkage hyperparameter is needed.

3. **Eigen and LSQR solvers require shrinkage** regularization to work at all on rank-deficient data. The choice of shrinkage level affects accuracy.

4. **Streaming `partial_fit` matches batch** accuracy for the SVD solver, enabling incremental face recognition where new subjects or images arrive over time without retraining from scratch.

5. **Practical takeaway:** When $N < D$ (common in image/genomics/spectral data), use `solver='svd'`. It is the only solver that works out of the box, and `partial_fit` now supports it for streaming updates.